# 3. LoRA fine-tuning OpenVLA-UAV

Notebook use LoRA (Low-Rank Adaptation) for OpenVLA droneactionfine-tuning. 

**target: **
- LoRA fine-tuning principle and advantages
- build UAV dataset and DataLoader
- configure and application LoRA adapter
- runtrainingloopmonitor
- savefine-tuning checkpoint

## 3.1 LoRA principle

all fine-tuning 7B need to large and. **LoRA** through in weight low implements high fine-tuning: 

```
weight: W (d x d)
LoRA: W' = W + B @ A its in A (r x d), B (d x r), r << d

: d=4096, r=32
parameter: 4096 x 4096 = 16.7M
LoRA parameter: 4096 x 32 + 32 x 4096 = 0.26M ( 1.6%)
```

advantages: 
- training ~1% parameter, large few 
- trainingvelocity fast, convergence also fast 
- inference LoRA weight, no outside 

## 3.2 environment

In [ ]:
import sys
import os
import json
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'disabled'

PROJECT_ROOT = "/root/autodl-tmp/claude/UAV-Flow/OpenVLA-UAV"
sys.path.insert(0, PROJECT_ROOT)

MODEL_PATH = "/root/autodl-tmp/claude/models/models--openvla--openvla-7b/snapshots/47a0ec7fc4ec123775a391911046cf33cf9ed83f"
DATA_DIR = "/root/autodl-tmp/claude/data/uav_flow_subset"
RUN_DIR = "/root/autodl-tmp/claude/runs/notebook_demo"
ADAPTER_DIR = "/root/autodl-tmp/claude/adapter-tmp/notebook_demo"

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f": {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 3.3 register & load Processor

In [ ]:
from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticProcessor, PrismaticImageProcessor
from transformers import AutoConfig, AutoImageProcessor, AutoProcessor, AutoModelForVision2Seq

AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f"Processor loaded. Vocab size: {processor.tokenizer.vocab_size}")

## 3.4 builddataset

`SimpleVLADataset` is UAV-Flow use datasetclass,: 
1. from loaddata
2. all convertcoordinate system below for action
3. calculateactionstatistics ( use normalization)
4. image + instruction + action format

In [ ]:
from prismatic.models.backbones.llm.prompting import PurePromptBuilder
from prismatic.vla.action_tokenizer import ActionTokenizer
from prismatic.vla.datasets.uav_dataset import SimpleVLADataset

action_tokenizer = ActionTokenizer(processor.tokenizer)

print("loaddataset...")
dataset = SimpleVLADataset(
data_path=DATA_DIR,
image_transform=processor.image_processor,
tokenizer=processor.tokenizer,
action_tokenizer=action_tokenizer,
prompt_builder_fn=PurePromptBuilder,
is_train=True,
)

print(f"\ndatasetloadcomplete!")
print(f": {len(dataset)}")
print(f"actionstatistics:")
print(f" mean: {dataset.action_mean}")
print(f" std: {dataset.action_std}")
print(f" min (1%): {dataset.action_min}")
print(f" max (99%): {dataset.action_max}")

In [ ]:
# 
sample_iter = iter(dataset)
sample = next(sample_iter)

print(" training:")
for key, val in sample.items():
if isinstance(val, torch.Tensor):
print(f" {key}: shape={val.shape}, dtype={val.dtype}")
else:
print(f" {key}: {val}")

# decoding label in action token
action_mask = sample['labels'] > action_tokenizer.action_token_begin_idx
action_ids = sample['labels'][action_mask].numpy()
print(f"\naction token: {len(action_ids)}")
if len(action_ids) > 0:
decoded = action_tokenizer.decode_token_ids_to_actions(action_ids)
print(f"decoding after action [-1,1]: {decoded}")

## 3.5 load & configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

# load
print("load OpenVLA-7B...")
vla = AutoModelForVision2Seq.from_pretrained(
MODEL_PATH,
torch_dtype=torch.bfloat16,
low_cpu_mem_usage=True,
trust_remote_code=True,
).to("cuda:0")

print(f"parameter: {sum(p.numel() for p in vla.parameters()) / 1e9:.2f}B")
print(f" use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# configure LoRA
lora_config = LoraConfig(
r=32, # LoRA ( large has, but parameter many)
lora_alpha=16, # LoRA 
lora_dropout=0.0, # Dropout 
target_modules="all-linear", # for has LoRA
init_lora_weights="gaussian", # initialize
)

print("LoRA configure:")
print(f" rank (r): {lora_config.r}")
print(f" alpha: {lora_config.lora_alpha}")
print(f" target_modules: {lora_config.target_modules}")

# application LoRA
vla = get_peft_model(vla, lora_config)
vla.print_trainable_parameters()

can, LoRA training **1%** parameter (~70M out of 7B), large few. 

## 3.6 build DataLoader

In [ ]:
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from torch.utils.data import DataLoader

# Collator: not long padding long 
collator = PaddedCollatorForActionPrediction(
model_max_length=processor.tokenizer.model_max_length,
pad_token_id=processor.tokenizer.pad_token_id,
padding_side="right",
)

# DataLoader
BATCH_SIZE = 4 # RTX 4080 SUPER 32GB use; OOM 2

dataloader = DataLoader(
dataset,
batch_size=BATCH_SIZE,
collate_fn=collator,
drop_last=True,
num_workers=2,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"DataLoader created.")

# verify batch
batch = next(iter(dataloader))
print(f"\nBatch content:")
for key, val in batch.items():
if isinstance(val, torch.Tensor):
print(f" {key}: shape={val.shape}, dtype={val.dtype}")

## 3.7 trainingloop

this inside training **50 ** (trainingneed to 500-5000). 

In [ ]:
from torch.optim import AdamW
from collections import deque

# trainingconfigure
MAX_STEPS = 50
LEARNING_RATE = 5e-4
GRAD_ACCUMULATION = 2 # has batch = 4 * 2 = 8

# optimizer
trainable_params = [p for p in vla.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LEARNING_RATE)

# record
history = {'step': [], 'loss': [], 'accuracy': [], 'l1_loss': []}

print(f"begintraining:")
print(f" Max steps: {MAX_STEPS}")
print(f" Batch size: {BATCH_SIZE} x {GRAD_ACCUMULATION} = {BATCH_SIZE * GRAD_ACCUMULATION}")
print(f" Learning rate: {LEARNING_RATE}")
print(f": {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("-" * 60)

In [ ]:
vla.train()
optimizer.zero_grad()
device_id = 0

for batch_idx, batch in enumerate(dataloader):
# Forward pass
with torch.autocast("cuda", dtype=torch.bfloat16):
output = vla(
input_ids=batch["input_ids"].to(device_id),
attention_mask=batch["attention_mask"].to(device_id),
pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
labels=batch["labels"],
)
loss = output.loss

# Backward
normalized_loss = loss / GRAD_ACCUMULATION
normalized_loss.backward()

# calculateactionaccuracy
action_logits = output.logits[:, vla.vision_backbone.featurizer.patch_embed.num_patches: -1]
action_preds = action_logits.argmax(dim=2)
action_gt = batch["labels"][:, 1:].to(action_preds.device)
mask = action_gt > action_tokenizer.action_token_begin_idx
correct_preds = (action_preds == action_gt) & mask
accuracy = correct_preds.sum().float() / mask.sum().float() if mask.sum() > 0 else torch.tensor(0.0)

# L1 loss
try:
pred_actions = torch.tensor(action_tokenizer.decode_token_ids_to_actions(action_preds[mask].cpu().numpy()))
gt_actions = torch.tensor(action_tokenizer.decode_token_ids_to_actions(action_gt[mask].cpu().numpy()))
l1_loss = torch.nn.functional.l1_loss(pred_actions, gt_actions).item()
except:
l1_loss = 0.0

# Optimizer step
gradient_step = batch_idx // GRAD_ACCUMULATION
if (batch_idx + 1) % GRAD_ACCUMULATION == 0:
optimizer.step()
optimizer.zero_grad()

# record
history['step'].append(gradient_step)
history['loss'].append(loss.item())
history['accuracy'].append(accuracy.item())
history['l1_loss'].append(l1_loss)

if gradient_step % 5 == 0:
print(f" Step {gradient_step:>3d} | loss={loss.item():.4f} | acc={accuracy.item():.3f} | l1={l1_loss:.4f}")

if gradient_step >= MAX_STEPS:
break

print("-" * 60)
print(f"trainingcomplete! {MAX_STEPS} ")
print(f" loss: {history['loss'][-1]:.4f}")
print(f" accuracy: {history['accuracy'][-1]:.3f}")

## 3.8 training

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['step'], history['loss'], 'b-')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['step'], history['accuracy'], 'g-')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Action Token Accuracy')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['step'], history['l1_loss'], 'r-')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('L1 Loss')
axes[2].set_title('Action L1 Loss')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Training Metrics ({MAX_STEPS} steps, LoRA r=32)', fontsize=14)
plt.tight_layout()
plt.show()

## 3.9 save Checkpoint

save LoRA adapter and after. 

In [ ]:
from prismatic.vla.datasets.uav_dataset import SimpleVLADataset

# save LoRA adapter
print("save LoRA adapter...")
processor.save_pretrained(RUN_DIR)
vla.save_pretrained(ADAPTER_DIR)

# saveactionstatistics (inferenceneed to)
norm_stats = {
"sim": {
"action": {
"mean": dataset.action_mean.tolist(),
"std": dataset.action_std.tolist(),
"min": dataset.action_min.tolist(),
"max": dataset.action_max.tolist(),
"q01": dataset.action_min.tolist(),
"q99": dataset.action_max.tolist(),
}
}
}
with open(os.path.join(RUN_DIR, "dataset_statistics.json"), 'w') as f:
json.dump(norm_stats, f, indent=2)

print(f"Adapter save: {ADAPTER_DIR}")
print(f"Processor save: {RUN_DIR}")
print(f"statisticssave: {RUN_DIR}/dataset_statistics.json")

In [ ]:
# () LoRA weight
from peft import PeftModel

print(" LoRA weight...")
base_vla = AutoModelForVision2Seq.from_pretrained(
MODEL_PATH, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
)
merged_vla = PeftModel.from_pretrained(base_vla, ADAPTER_DIR)
merged_vla = merged_vla.merge_and_unload()
merged_vla.save_pretrained(RUN_DIR)

print(f" after save: {RUN_DIR}")
print(f"content: {os.listdir(RUN_DIR)}")

In [ ]:
# 
del vla, base_vla, merged_vla
torch.cuda.empty_cache()
print(".")

## small 

- **LoRA** training ~1% parameter, 32GB GPU i.e.fine-tuning 7B 
- trainingdata: image + instruction → Processor → before toward → action token loss
- key: **loss** ( low good), **action accuracy** (token), **L1 loss** (actionerror)
- 50 is, training 500-5000 ( for `bash run_finetune.sh`)
- save need save **weight** and **actionstatistics** (dataset_statistics.json)

** below:** in Notebook 04 in, loadfine-tuning after inference. 